In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
import numpy as np
import math
import seaborn as sns
import pyarrow as pa
import pyarrow.parquet as pq
import itertools
import evi_functions as evi_func
import matplotlib.dates as mdates
import warnings
from pathlib import Path
from datetime import datetime
import descri_function as des_fun
import scipy.stats as stats

# Read data

In [2]:
#dados das replicas

df = pd.read_parquet('/opt/storage/shared/aesop/aesop_shared/ensamble_modelling/sintetic_with_MEM_surge_col_29_04_2026.parquet')

In [3]:
# get parameters
folder = Path("/opt/storage/raw/aesop/visualization/ensamble_modelling/evi_par/")

evi_par = pd.concat(
    [pd.read_parquet(f) for f in folder.glob("*.parquet")],
    ignore_index=True
)


In [4]:
evi_par = evi_par.assign(co_ibge = evi_par.co_ibge.astype(int),
                                   m = evi_par.m.astype(int))

# Run evi with optimised parameters for all city and each replicate

In [5]:
evi_par_dict = evi_par.set_index("co_ibge")[["m","c"]].to_dict("index")

def evi_sint(df, series_list):

    lst_dfs = []

    for code in df.co_ibge.unique():

        set_muni = df[df.co_ibge == code].copy()

        pars = evi_par_dict.get(code)
        if pars is None:
            m_, c_ = 5, 0.2
        else:
            m_, c_ = pars["m"], pars["c"]

        set_muni = set_muni.reset_index(drop=True)

        for serie in series_list:

            dtf = evi_func.func(m_, set_muni[serie].to_numpy(), 2, 8, c_)

            col_evi = f"evi_t1_t_{serie}"
            col_ind = f"sinal_evi_{serie}"

            set_muni[col_evi] = dtf["evi_t1_t"]
            set_muni[col_ind] = dtf["ind"].astype(int)

            # clean infinities
            finite_vals = set_muni.loc[np.isfinite(set_muni[col_evi]), col_evi]

            if len(finite_vals) > 0:
                max_value = np.nanmax(finite_vals)
                set_muni[col_evi] = set_muni[col_evi].replace([np.inf, -np.inf], max_value)

        lst_dfs.append(set_muni)

    return pd.concat(lst_dfs, ignore_index=True)

In [6]:
series_list = [col for col in df.columns if col.startswith("replicate_")]

In [7]:
df_out = evi_sint(df, series_list)

In [8]:
df_out.columns.to_list()

['replicate_0',
 'replicate_1',
 'replicate_2',
 'replicate_3',
 'replicate_4',
 'replicate_5',
 'replicate_6',
 'replicate_7',
 'replicate_8',
 'replicate_9',
 'replicate_10',
 'replicate_11',
 'replicate_12',
 'replicate_13',
 'replicate_14',
 'replicate_15',
 'replicate_16',
 'replicate_17',
 'replicate_18',
 'replicate_19',
 'replicate_20',
 'replicate_21',
 'replicate_22',
 'replicate_23',
 'replicate_24',
 'replicate_25',
 'replicate_26',
 'replicate_27',
 'replicate_28',
 'replicate_29',
 'replicate_30',
 'replicate_31',
 'co_ibge',
 'year_week',
 'atend_ivas',
 'mem_surge_01_correct_with_consec',
 'warning_final_mem_surge_01',
 'mem_surge_01_replicate_0',
 'mem_surge_01_replicate_0_without_isolated',
 'mem_surge_01_replicate_0_correct_with_consec',
 'warning_final_mem_surge_01_replicate_0',
 'mem_surge_01_replicate_1',
 'mem_surge_01_replicate_1_without_isolated',
 'mem_surge_01_replicate_1_correct_with_consec',
 'warning_final_mem_surge_01_replicate_1',
 'mem_surge_01_replicat

In [10]:
from pathlib import Path
from datetime import datetime

out_dir = Path("/opt/storage/shared/aesop/aesop_shared/ensamble_modelling")

fname = f"sintetic_evi_{datetime.now():%d_%m_%Y}.parquet"

df_out.to_parquet(out_dir / fname)